# Ramsey demo (tProc v2) with the current lab configuration

A standalone copy of **only the Ramsey section** of `02_qubit_demos.ipynb`, with the
config dictionary filled from `qick_tprocv2_experiments/system_config.json` for
**Q1** as of 2026-09-17, and the board connection changed to run **directly on the
RFSoC** (no Pyro).

The point is a clean comparison against the experiment scripts: this is the demo's
`RamseyProgram` as written, so any difference in the fringe phase between this and
`006_Ramsey_ge.py` comes from the things the two do differently, not from the config.

Deviations from the demo, each marked `[n]` in the code:

1. **Readout is tProc-configured** (`add_readoutconfig` / `send_readoutconfig`) instead
   of the demo's pynq-configured `declare_readout(..., freq=, gen_ch=)`. Readout channel
   0 is a *dynamic* readout in this firmware and only accepts the former - the
   demo carries both styles with one commented out, so this is just flipping that switch.
2. **`nqz` comes from the config** rather than being hardcoded to 1, because the readout
   here is at 7202.5 MHz on a full-speed DAC.
3. **The phase advance uses `ramsey_freq` from the config** in place of the demo's
   hardcoded `2*360*(0.01)`, matching the expression in `qick_setup_funcs.declare_pulse`.

Everything else - `ro_ch` on the qubit pulses, the `sigma*4` envelope, no gap between the
second pulse and the readout, `wait_auto` in the body - is the demo's.

Imports

In [ ]:
# jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *

# for now, all the tProc v2 classes need to be individually imported (can't use qick.*)

# the main program class
from qick.asm_v2 import AveragerProgramV2
# for defining sweeps
from qick.asm_v2 import QickSpan, QickSweep1D

Connect to the RFSoC **directly** - this notebook runs on the board itself, not through Pyro.

In [ ]:
# Loads the firmware and talks to the hardware in-process. QickSoc is also the
# QickConfig, so the same object serves as soccfg.
soc = QickSoc()
# If the board's default bitfile is not the one you want, name it explicitly:
# soc = QickSoc(bitfile='/path/to/qick_216_standard.bit')
soccfg = soc
print(soccfg)

# Ramsey Fringe Experiment

Measures the qubit T2 Ramsey characteristic decay time by preparing the qubit in a superposition state with a  $\frac{\pi}{2}$ pulse followed by a variable delay ($\tau$) before applying another $\frac{\pi}{2}$ pulse with a phase advanced by $\Delta \phi = \omega * \tau$, where $\omega$ is the Ramsey frequency. This exponential decay envelope can be fitted in post-processing to obtain the qubit T2 Ramsey time.

In [ ]:
class RamseyProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        res_ch = cfg['res_ch']
        qubit_ch = cfg['qubit_ch']

        # [2] nqz from the config instead of the demo's hardcoded 1
        self.declare_gen(ch=res_ch, nqz=cfg['nqz_res'])
        self.declare_gen(ch=qubit_ch, nqz=cfg['nqz_qubit'])

        # [1] pynq-configured readout, as the demo has it - does not work on a
        #     dynamic (tProc-configured) readout channel:
        # self.declare_readout(ch=ro_ch, length=cfg['ro_len'], freq=cfg['f_res'], gen_ch=res_ch)
        # [1] tProc-configured readout, the demo's commented-out alternative:
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['f_res'], gen_ch=res_ch)

        self.add_loop("waitloop", cfg["steps"])

        self.add_pulse(ch=res_ch, name="res_pulse", ro_ch=ro_ch,
                       style="const",
                       length=cfg['res_len'],
                       freq=cfg['f_res'],
                       phase=cfg['res_phase'],
                       gain=cfg['res_gain'],
                      )

        self.add_gauss(ch=qubit_ch, name="ramp", sigma=cfg['sigma'], length=cfg['sigma']*4, even_length=True)
        self.add_pulse(ch=qubit_ch, name="qubit_pulse1", ro_ch=ro_ch,
                       style="arb",
                       envelope="ramp",
                       freq=cfg['f_ge'],
                       phase=cfg['qubit_phase'],
                       gain=cfg['pi2_gain'],
                      )

        self.add_pulse(ch=qubit_ch, name="qubit_pulse2", ro_ch=ro_ch,
                       style="arb",
                       envelope="ramp",
                       freq=cfg['f_ge'],
                       # [3] current phase + time * 360 * ramsey freq (the demo hardcodes 2*360*(0.01))
                       phase=cfg['qubit_phase'] + cfg['wait_time']*360*cfg['ramsey_freq'],
                       gain=cfg['pi2_gain'],
                      )

    def _body(self, cfg):
        # [1] tProc-configured readout needs the config sent each shot
        self.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)
        self.pulse(ch=self.cfg["qubit_ch"], name="qubit_pulse1", t=0)  #play pulse

        self.delay_auto(cfg['wait_time'], tag='wait') # wait_time after last pulse

        self.pulse(ch=self.cfg["qubit_ch"], name="qubit_pulse2", t=0)  #play pulse

        self.pulse(ch=cfg['res_ch'], name="res_pulse", t=0) # play probe pulse
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])

        self.wait_auto(cfg['relax_delay']) # wait for qubit to relax

Configuration - every value below is from `system_config.json`, Q1, 2026-09-17. Edit here, not in the class.

In [ ]:
START_TIME = 0.0 # [us]      Ramsey_ge start
STOP_TIME = 1 # [us]        Ramsey_ge stop[Q1]
STEPS = 200                 # Ramsey_ge steps

config = {
    ## Sweep Params: ##
    'steps': STEPS, # Number of sweep points

    ## Channel Params. ##
    'res_ch': 1,   # hw_cfg res_ch[Q1]
    'ro_ch': 0,    # hw_cfg ro_ch[Q1]
    'qubit_ch': 0, # hw_cfg qubit_ch[Q1]
    'nqz_res': 2,  # hw_cfg nqz_res
    'nqz_qubit': 1, # hw_cfg nqz_qubit

   ## Resonator Pulse Params. ##
    'f_res': 7202.525616, # [MHz]   readout_cfg res_freq_ge[Q1]
    'res_len': 5.5, # [us]           readout_cfg res_length_ge
    'res_phase': -133.6, # [deg]       readout_cfg res_phase[Q1]
    'res_gain': 0.05, # [DAC units]   readout_cfg res_gain_ge[Q1]

    ## Readout Params. ##
    'trig_time': 0.55, # [us]        readout_cfg trig_time
    'ro_len': 5.4, # [us]            readout_cfg ro_length
    'relax_delay': 100, # [us]      Ramsey_ge relax_delay
    'wait_time': QickSweep1D('waitloop', START_TIME, STOP_TIME), # [us]
    'ramsey_freq': 3.0, # [MHz]     Ramsey_ge ramsey_freq

    ## Qubit Params. ##
    'f_ge': 5069.640336, # [MHz]  qubit_cfg qubit_freq_ge[Q1]
    "pi_gain": 0.118, # [DAC units]     qubit_cfg qubit_gain_ge[Q1]
    "pi2_gain": 0.057, # [DAC units]    qubit_cfg half_pi_gain_ge[Q1]
    'qubit_phase': 0,              # qubit_cfg qubit_phase
    'sigma': 0.03, # [us]               qubit_cfg sigma_ge[Q1]
     }

In [ ]:
ramsey=RamseyProgram(soccfg, reps=100, final_delay=0.5, cfg=config)
iq_list = ramsey.acquire(soc, soft_avgs=10, progress=True)
delay_times = ramsey.get_time_param('wait', "t", as_array=True)

plt.subplot(111,title=f"Ramsey Fringe Experiment", xlabel="Delay Time (us)", ylabel="Qubit Population")
plt.plot(delay_times, np.abs(iq_list[0][0].dot([1,1j])),'o-')

Timing diagnostics - what the tProc actually did with the requested sweep.

In [ ]:
step_actual_ns = np.diff(delay_times).mean() * 1e3
step_nominal_ns = (STOP_TIME - START_TIME) * 1e3 / (STEPS - 1)
print(f"first delay point:    {delay_times[0]*1e3:.3f} ns  (tProc time at the end of pulse 1)")
print(f"requested step:       {step_nominal_ns:.4f} ns")
print(f"actual step:          {step_actual_ns:.4f} ns")
print(f"phase/time mismatch:  x{step_nominal_ns/step_actual_ns:.4f}  "
      "(the phase advance per step is computed from the requested step, not the actual one)")

Optional - save in the same layout `006_Ramsey_ge.py` writes, so `analyze_t2_ramsey`
on the PC can read it back. Uses plain `h5py`, since `malab` is not on the board.

In [ ]:
import h5py, json, datetime, os

amps = np.abs(iq_list[0][0].dot([1,1j]))
avgi = iq_list[0][0].T[0]
avgq = iq_list[0][0].T[1]

# the keys analyze_t2_ramsey reads from the config attr, under the names it expects
saved_config = {k: val for k, val in config.items() if k != 'wait_time'}   # QickSweep1D is not JSON
saved_config.update({
    'qubit_freq_ge': config['f_ge'],
    'sigma_ge': config['sigma'],
    'half_pi_gain_ge': config['pi2_gain'],
    'qubit_gain_ge': config['pi_gain'],
    'ro_length': config['ro_len'],
    'res_length_ge': config['res_len'],
    'start': START_TIME, 'stop': STOP_TIME,
    'qubit_index': 1,
    'source': 'ramsey demo (v2) run directly on the board',
})

i = 0
fname = f"{i:05d}_Ramsey_demo_v2_Q1_{datetime.date.today()}.h5"
while os.path.exists(fname):
    i += 1
    fname = f"{i:05d}_Ramsey_demo_v2_Q1_{datetime.date.today()}.h5"

with h5py.File(fname, 'w') as f:
    # SlabFile.append writes each dataset as one row of a 2D array; the
    # analysis reads array(f[...])[0], so match that shape.
    for name, arr in [('delay_times', delay_times), ('amps', amps), ('avgi', avgi), ('avgq', avgq)]:
        f.create_dataset(name, data=np.asarray(arr)[None, :])
    f.attrs['config'] = json.dumps(saved_config)
print('saved', fname)